# Loan Default Predictor — EDA

Explores the UCI German Credit dataset.

**Note:** Feature pipeline is not re-implemented here.
It comes from Mlops-Plumbing:
```python
from src.features.transformers import build_feature_pipeline
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from omegaconf import OmegaConf

# Import feature pipeline directly from Mlops-Plumbing git dependency
from src.features.transformers import build_feature_pipeline

print('Mlops-Plumbing feature pipeline imported successfully.')

In [ ]:
cfg = OmegaConf.load('../configs/default.yaml')

try:
    df = pd.read_csv('../data/raw/loan_data.csv')
    print(f'Loaded {len(df):,} rows from local file')
except FileNotFoundError:
    from src.data.ingest import COLUMN_NAMES, download, parse_raw
    from pathlib import Path
    dest = Path('/tmp/german.data')
    download(cfg.data.source_url, dest)
    df = parse_raw(dest)
    print(f'Downloaded {len(df):,} rows')

df.head()

## Default Rate

In [ ]:
default_rate = df['default'].mean()
print(f'Default rate: {default_rate:.1%}')

fig = px.pie(
    values=df['default'].value_counts().values,
    names=['No Default', 'Default'],
    title=f'Default Distribution ({default_rate:.1%} default rate)',
    color_discrete_sequence=['#2196F3', '#FF5252'],
    hole=0.4,
)
fig.show()

## Numeric Features by Default Status

In [ ]:
numeric_cols = ['duration', 'credit_amount', 'age']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, numeric_cols):
    for label, grp in df.groupby('default'):
        ax.hist(grp[col], bins=30, alpha=0.6,
                label='Default' if label == 1 else 'No Default',
                color='#FF5252' if label == 1 else '#2196F3')
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

## Feature Pipeline Demo (from Mlops-Plumbing)

In [ ]:
# Demonstrate that build_feature_pipeline() from Mlops-Plumbing works
# on this project's data — zero custom pipeline code needed here.

X = df.drop(columns=['default'])
y = df['default']

pipeline = build_feature_pipeline(cfg)  # ← directly from Mlops-Plumbing
X_transformed = pipeline.fit_transform(X)

print(f'Original shape:    {X.shape}')
print(f'Transformed shape: {X_transformed.shape}')
print(f'\nPipeline steps: {list(pipeline.named_steps.keys())}')

## Key Takeaways

1. **Default rate ~30%** — class imbalance, SMOTE justified.
2. **Longer duration** loans default more.
3. **Higher credit amounts** correlate with default.
4. **Younger applicants** default at higher rates.
5. **Checking account status** is the strongest categorical predictor.
6. **Mlops-Plumbing's `build_feature_pipeline()`** handles all encoding/scaling — no boilerplate needed in this repo.